# Uczenie Maszynowe w praktyce - Python - PART 1.2 - regresja

## 1. Import Bibliotek

In [ ]:
# install package in anaconda prompt
# pip install rdatasets
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

## KROK 1: Import Danych

In [ ]:
try:
    insurance = pd.read_csv("data/insurance.csv")
except FileNotFoundError:
    try:
        insurance = pd.read_csv("../data/insurance.csv")
    except FileNotFoundError:
        print("Nie znaleziono pliku. Sprawdź ścieżkę.")

## KROK 2: Przygotowanie Danych - Data wrangling

In [ ]:
insurance['bmi_above_30'] = np.where(insurance['bmi'] > 30 , 1, 0)

### Wizualizacja Danych

In [ ]:
corr = insurance.corr(method="pearson", numeric_only=True)
sns.pairplot(insurance)
plt.show()

In [ ]:
plt.subplots(figsize=(8,6))
sns.scatterplot(x='age', y='charges', data=insurance, hue='smoker')
plt.show()

In [ ]:
plt.subplots(figsize=(8,6))
sns.scatterplot(x='age', y='charges', data=insurance, hue='bmi_above_30')
plt.show()

## KROK 4: Uczenie Modelu

In [ ]:
model1 = smf.ols(
    formula = "charges ~ age + C(sex) + bmi + children + C(smoker) + C(region) + C(bmi_above_30)", 
    data=insurance
)
result1 = model1.fit()

model2 = smf.ols(
    formula = "charges ~ age + bmi + C(sex)  + children + smoker * bmi_above_30", 
    data=insurance
)
result2 = model2.fit()

## KROK 5: Ocena Wyników

In [ ]:
print(result1.summary())

In [ ]:
print(result2.summary())

### Analiza Modelu 1

In [ ]:
forecast_model1 = result1.fittedvalues

fig = plt.figure(figsize=(8,6))
sm.graphics.plot_regress_exog(result1, "age", fig=fig)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sm.graphics.plot_fit(result1, "age", ax=ax)
plt.show()

### Histogram reszt (Model 1)

In [ ]:
plt.subplots(figsize=(8,6))
result1.resid.hist()
plt.show()

### Analiza Modelu 2

In [ ]:
forecast_model2 = result2.fittedvalues

### Testy Normalności (Shapiro-Wilk)

In [ ]:
from scipy.stats import shapiro

# H0 - is normal p > 0.05
# HA - is not normal  p < 0.05

stat1, p1 = shapiro(result1.resid)
print(f"Shapiro p-value: {p1}")

### Porównanie Metryk (Metrics)

In [ ]:
from statsmodels.tools import eval_measures
y = insurance['charges']

def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("\n" + "="*50)
print(f"{'Metryka':<15} | {'Model 1':<15} | {'Model 2':<15}")
print("-" * 50)
print(f"{'R-squared':<15} | {result1.rsquared:<15.4f} | {result2.rsquared:<15.4f}")
print(f"{'RMSE':<15} | {eval_measures.rmse(y, forecast_model1):<15.4f} | {eval_measures.rmse(y, forecast_model2):<15.4f}")
print(f"{'MAPE (%)':<15} | {mean_absolute_percentage_error(y, forecast_model1):<15.4f} | {mean_absolute_percentage_error(y, forecast_model2):<15.4f}")
print(f"{'AIC':<15} | {result1.aic:<15.4f} | {result2.aic:<15.4f}")
print(f"{'BIC':<15} | {result1.bic:<15.4f} | {result2.bic:<15.4f}")
print("="*50 + "\n")

if result2.rsquared > result1.rsquared and result2.aic < result1.aic:
    print("Wniosek: Model 2 ma lepsze parametry dopasowania.")
else:
    print("Wniosek: Niejednoznaczne wyniki lub Model 1 jest lepszy.")

## KROK 6: Predykcja dla nowych danych

In [ ]:
print("\n" + "="*90)
print("PREDYKCJA KOSZTÓW DLA RÓŻNYCH PACJENTÓW (PORÓWNANIE)")
print("-" * 90)

# 1. Definiujemy dane dla kilku pacjentów o różnych profilach
patients_data = {
    'opis': ['Młody, zdrowy, student', 'Starsza Pani, zdrowa', 'Otyły, niepalący', 'Palacz, szczupły', 'Palacz, otyły (High Risk)'],
    'age': [21, 62, 30, 30, 30],
    'sex': ['male', 'female', 'male', 'male', 'male'],
    'bmi': [22.5, 26.0, 36.5, 23.5, 36.5],
    'children': [0, 1, 1, 0, 0],
    'smoker': ['no', 'no', 'no', 'yes', 'yes'],
    'region': ['northwest', 'southeast', 'southwest', 'northeast', 'southeast']
}

patients_df = pd.DataFrame(patients_data)

# 2. Feature Engineering - to samo co przy uczeniu!
patients_df['bmi_above_30'] = np.where(patients_df['bmi'] > 30 , 1, 0)

# 3. Predykcja
patients_df['predicted_cost'] = result2.predict(patients_df)

# Wyświetlenie wyników w formie tabeli
print(f"{'Opis pacjenta':<30} | {'Wiek':<5} | {'BMI':<5} | {'Palacz':<6} | {'Przewidywany Koszt ($)':<25}")
print("-" * 90)

for index, row in patients_df.iterrows():
    print(f"{row['opis']:<30} | {row['age']:<5} | {row['bmi']:<5} | {row['smoker']:<6} | {row['predicted_cost']:<10.2f}")

print("="*90 + "\n")